In [ ]:
import torch
import torchvision

/home/zhang402/miniconda3/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initialize DNN model and compile using TorchInductor

In [2]:
model = torchvision.models.inception_v3(pretrained=True).to("cuda")

/home/zhang402/miniconda3/envs/myenv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/zhang402/miniconda3/envs/myenv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [3]:
torch._dynamo.reset()
inceptionv3_compiled = torch.compile(
        model,
        options={
            "trace.enabled": True,
        },
)

Set up the training loop

In [4]:
# for this example, we generate one random sample
inputs = torch.randn(32, 3, 299, 299).to("cuda")
labels = torch.randn(32, 1000).to("cuda")

# initialize the loss calculation and optimizer
learning_rate = 0.001
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(inceptionv3_compiled.parameters(), lr=learning_rate)

Wrap optimizer.step() in torch.compile()

In [5]:
def optimizer_step_fn(optimizer):
    '''Return torch.compile'd version of optimizer.step()'''
    def f():
        optimizer.step()
    return torch.compile(
        f,
        options={
            "trace.enabled": True,
        },
    )

optimizer_step = optimizer_step_fn(optimizer)

Run one training iteration

In [6]:
# Zero out the optimizer
optimizer.zero_grad()

# Forward pass
outputs, _ = inceptionv3_compiled(inputs)
loss = criterion(outputs, labels) # torch.nn.CrossEntropyLoss()
# outputs, loss = forward(inputs, labels)

# Backward pass
loss.backward()

# parameter update
optimizer_step()

/home/zhang402/miniconda3/envs/myenv/lib/python3.10/site-packages/torch/_inductor/compile_fx.py:124: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
W0514 22:07:37.955000 140545960215744 torch/_inductor/debug.py:413] [0/0] model__0_forward_10 debug trace: /tmp/torchinductor_zhang402/xu/cxuyj6xihv4lwis2ljxklh4pk6m6qyjlqjbhlnmdqb2j65c4exa2.debug
W0514 22:08:16.549000 140541948004096 torch/_inductor/debug.py:413] model__0_backward_12 debug trace: /tmp/torchinductor_zhang402/au/cauudcwvbymmdbuz2tmiy5b3lbae3aw47bmg3zbnsnc666frf7ji.debug
W0514 22:08:28.796000 140545960215744 torch/_logging/_internal.py:1013] [1/0] Profiler function <class 'torch.autograd.profiler.record_function'> will be ignored
W0514 22:09:13.333000 140545960215744 torch/_inductor/debug.py:413] [3/0] model__4_inference_13 debug trace: /tmp/torchinductor_zhang402/vy/cvye

Example operator profiling loop

In [7]:
import time
num_iter = 1000
device = "cuda"

# allocate dummy inputs
primals_321 = torch.randn(64, 3, 224, 224).to(device)
primals_1 = torch.randn(64, 3, 7, 7).to(device)

t0 = time.time()

# profile the operator
for _ in range(num_iter):
    convolution = torch.ops.aten.convolution.default(primals_321, primals_1, None, [2, 2], [3, 3], [1, 1], False, [0, 0], 1)
torch.cuda.current_stream().synchronize()

t1 = time.time()

print(f"Time taken: {(t1 - t0) / num_iter * 1000} ms")

Time taken: 1.1168503761291504 ms
